4. Rank best destinations
5. Plotly map -1

In [ ]:
"""
The weather score is designed to favor destinations with comfortable daytime temperatures, low rainfall, 
low probability of precipitation, moderate wind, and moderate humidity. Temperature is given the highest 
weight because it has the strongest impact on travel comfort, while rainfall and precipitation probability 
are the main negative factors.


Scoring system:

Average daytime temperature
Total rain over 7 days
Average probability of precipitation (pop)
Average wind speed
Average humidity

Weather Score
=
+ Temperature Score
- Rain Penalty
- Wind Penalty
- Humidity Penalty


"""

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

weather_df = pd.read_csv(
    RAW_DATA_DIR / "weather_daily_7days.csv"
)

weather_df.head()

,city_id,city,latitude,longitude,date,temp_day,temp_min,temp_max,feels_like_day,humidity,weather,clouds,wind_speed,pop,rain
0,1,Mont Saint Michel,48.635954,-1.51146,2026-08-15,24.30,17.98,25.36,24.30,64,light rain,100,6.62,0.8,3.16
1,1,Mont Saint Michel,48.635954,-1.51146,2026-08-16,25.98,16.96,26.20,25.98,55,broken clouds,72,6.87,0.0,0.00
2,1,Mont Saint Michel,48.635954,-1.51146,2026-08-17,22.32,16.11,23.45,22.32,61,overcast clouds,97,5.92,0.0,0.00
3,1,Mont Saint Michel,48.635954,-1.51146,2026-08-18,22.72,16.53,24.38,22.72,66,broken clouds,75,6.08,0.0,0.00
4,1,Mont Saint Michel,48.635954,-1.51146,2026-08-19,20.18,16.18,21.96,20.18,62,overcast clouds,99,4.99,0.0,0.03


In [3]:
# Aggregate the 7 days for each city

city_weather = (
    weather_df
    .groupby(
        ["city_id", "city", "latitude", "longitude"],
        as_index=False
    )
    .agg(
        avg_temp_day=("temp_day", "mean"),
        avg_temp_min=("temp_min", "mean"),
        avg_temp_max=("temp_max", "mean"),
        avg_humidity=("humidity", "mean"),
        avg_wind_speed=("wind_speed", "mean"),
        avg_pop=("pop", "mean"),
        total_rain=("rain", "sum")
    )
)

city_weather.head()

,city_id,city,latitude,longitude,avg_temp_day,avg_temp_min,avg_temp_max,avg_humidity,avg_wind_speed,avg_pop,total_rain
0,1,Mont Saint Michel,48.635954,-1.511460,22.175714,16.080000,23.435714,60.428571,6.042857,0.171429,15.06
1,2,St Malo,48.649518,-2.026041,20.412857,17.190000,21.410000,69.285714,6.770000,0.141429,8.39
2,3,Bayeux,49.276462,-0.702474,22.507143,15.065714,23.762857,55.714286,5.570000,0.240000,15.53
3,4,Le Havre,49.493898,0.107973,20.607143,18.517143,21.661429,67.285714,6.511429,0.188571,7.59
4,5,Rouen,49.440459,1.093966,24.384286,15.421429,27.242857,45.428571,6.204286,0.197143,8.06


In [4]:
city_weather.shape

(35, 11)

Rules:

ideal daytime temperature: around 24°C
less rain is better
lower precipitation probability is better
lower wind is better
moderate humidity is better

In [5]:
# Temperature

"""24°C  → 100 points
22°C  → 90 points
20°C  → 80 points
30°C  → 70 points"""

IDEAL_TEMP = 24

city_weather["temperature_score"] = (
    100 - abs(city_weather["avg_temp_day"] - IDEAL_TEMP) * 5
)

city_weather["temperature_score"] = (
    city_weather["temperature_score"].clip(lower=0, upper=100)
)


In [ ]:
# Penalties

# Rain
city_weather["rain_penalty"] = (
    city_weather["total_rain"] * 2
).clip(upper=100)

# Probability of precipitation
# pop is between 0 and 1, so 0.40 means roughly 40% precipitation probability
# Probability of precipitation (PoP) means the chance that measurable precipitation (rain, snow, etc.)
# will occur at a particular location during a specified period.

city_weather["pop_penalty"] = (
    city_weather["avg_pop"] * 100
)

# Wind

city_weather["wind_penalty"] = (
    city_weather["avg_wind_speed"] * 5
).clip(upper=100)

# Humidity, we'll penalize values that move too far away from a comfortable target of roughly 60%

IDEAL_HUMIDITY = 60

city_weather["humidity_penalty"] = (
    abs(city_weather["avg_humidity"] - IDEAL_HUMIDITY) * 1.5
).clip(upper=100)

In [7]:
# Score
# weighting temperature the most, then rain/precipitation, then wind and humidity

city_weather["weather_score"] = (
    0.40 * city_weather["temperature_score"]
    - 0.25 * city_weather["rain_penalty"]
    - 0.20 * city_weather["pop_penalty"]
    - 0.10 * city_weather["wind_penalty"]
    - 0.05 * city_weather["humidity_penalty"]
)

In [8]:
# Sort the cities
city_weather = city_weather.sort_values(
    "weather_score",
    ascending=False
)

# Check
city_weather[
    [
        "city",
        "avg_temp_day",
        "total_rain",
        "avg_pop",
        "avg_wind_speed",
        "avg_humidity",
        "weather_score"
    ]
].head(10)

,city,avg_temp_day,total_rain,avg_pop,avg_wind_speed,avg_humidity,weather_score
4,Rouen,24.384286,8.06,0.197143,6.204286,45.428571,27.063571
5,Paris,25.417143,5.19,0.234286,4.685714,41.857143,26.181429
34,La Rochelle,22.961429,6.71,0.291429,6.921429,62.714286,25.075000
6,Amiens,24.185714,9.92,0.271429,6.568571,44.428571,24.787857
0,Mont Saint Michel,22.175714,15.06,0.171429,6.042857,60.428571,22.339286
3,Le Havre,20.607143,7.59,0.188571,6.511429,67.285714,21.845714
1,St Malo,20.412857,8.39,0.141429,6.770000,69.285714,21.720714
2,Bayeux,22.507143,15.53,0.240000,5.570000,55.714286,21.342857
7,Lille,22.810000,10.93,0.425714,5.941429,48.571429,19.812857
20,Marseille,29.622857,6.43,0.184286,6.948571,51.571429,17.747143


In [9]:
# create your Top 5

top_5_destinations = city_weather.head(5).copy()

top_5_destinations[
    [
        "city_id",
        "city",
        "latitude",
        "longitude",
        "avg_temp_day",
        "total_rain",
        "avg_pop",
        "weather_score"
    ]
]

,city_id,city,latitude,longitude,avg_temp_day,total_rain,avg_pop,weather_score
4,5,Rouen,49.440459,1.093966,24.384286,8.06,0.197143,27.063571
5,6,Paris,48.853495,2.348391,25.417143,5.19,0.234286,26.181429
34,35,La Rochelle,46.159732,-1.151595,22.961429,6.71,0.291429,25.075000
6,7,Amiens,49.894171,2.295695,24.185714,9.92,0.271429,24.787857
0,1,Mont Saint Michel,48.635954,-1.511460,22.175714,15.06,0.171429,22.339286


In [10]:
# Save both the full ranking and the Top 5:

city_weather.to_csv(
    PROCESSED_DATA_DIR / "city_weather_ranking.csv",
    index=False
)

top_5_destinations.to_csv(
    PROCESSED_DATA_DIR / "top_5_destinations.csv",
    index=False
)